# Figuras del Capítulo 4 — Compresión SVD

Genera todas las figuras en **castellano** e **inglés** automáticamente.

- `figures/cap4_*_es.png` — versión castellano (para la memoria)
- `figures/cap4_*_en.png` — versión inglés (para futura traducción)

**Antes de ejecutar**: ajusta las rutas en la celda de configuración.

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Ajusta estas rutas a tu entorno
# ══════════════════════════════════════════════════════════════

NB02_DIR = "results/notebook2/"
NB03_DIR = "results/notebook3/"

# CSVs de NB02
CSV_K95_MATRIX       = NB02_DIR + "rank_matrix_k95.csv"             # 12×6: capas × componentes
CSV_SINGULAR_VALUES  = NB02_DIR + "singular_values_by_component.csv" # curvas de decaimiento
CSV_UNIFORM_RESULTS  = NB02_DIR + "uniform_compression_results.csv"  # F1 macro por rango
CSV_UNIFORM_EMOTIONS = NB02_DIR + "uniform_emotion_f1.csv"           # F1 por emoción × rango
CSV_ADAPTIVE_RESULTS = NB02_DIR + "adaptive_compression_results.csv"
CSV_ADAPTIVE_RANKS   = NB02_DIR + "adaptive_rank_distribution.csv"

# CSVs de NB03
CSV_COMP_SENSITIVITY  = NB03_DIR + "component_sensitivity.csv"
CSV_DEPTH_SENSITIVITY = NB03_DIR + "depth_sensitivity.csv"

# CSV de NB09
CSV_COMPRESSION_COMPARISON = "results/notebook9/compression_comparison.csv"

import os
os.makedirs("figures", exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tfg_plot_style as sty
sty.apply(lang="es")

# Helper to load CSV with fallback
def load_csv(path, fallback=None, name=""):
    try:
        df = pd.read_csv(path)
        print(f"✓ {name}: {path} — {df.shape}")
        return df
    except FileNotFoundError:
        if fallback is not None:
            print(f"⚠ {name}: {path} no encontrado — usando fallback")
            return fallback
        print(f"✗ {name}: {path} no encontrado — saltando")
        return None

---
## Fig. 1 — Anatomía arquitectónica: heatmap de $k_{95}$

**CSV necesario**: `rank_matrix_k95.csv` (12 filas × 6 cols: Query, Key, Value, Attn_Output, FFN_Inter, FFN_Output)

In [ ]:
k95_df = load_csv(CSV_K95_MATRIX, name="k95 matrix")

if k95_df is not None:
    def plot_anatomy(ax, df=k95_df):
        if isinstance(df.iloc[:, 0].dtype, object) or df.columns[0].lower() in ('layer', 'capa'):
            data = df.set_index(df.columns[0]).values
            comp_names = list(df.columns[1:])
        else:
            data = df.values
            comp_names = list(df.columns)
        n_layers = data.shape[0]

        im = ax.imshow(data, cmap=sty.CMAP_COOL, aspect="auto", vmin=150, vmax=700)
        ax.set_xticks(range(len(comp_names)))
        ax.set_xticklabels(comp_names, fontsize=sty.TICK_SIZE)
        ax.set_yticks(range(n_layers))
        ax.set_yticklabels([f"L{i}" for i in range(n_layers)], fontsize=sty.TICK_SIZE)
        ax.set_ylabel(sty.L["layer"])
        ax.set_xlabel(sty.L["component"])
        ax.set_title(sty.L["t_anatomy"])

        for i in range(n_layers):
            for j in range(len(comp_names)):
                val = int(data[i, j])
                color = "white" if val > 500 else sty.INK
                ax.text(j, i, str(val), ha="center", va="center",
                        fontsize=sty.SMALL_SIZE, color=color)

        # Divider
        ffn_idx = next((i for i, c in enumerate(comp_names) if "FFN" in c or "ffn" in c), 4)
        ax.axvline(x=ffn_idx - 0.5, color=sty.INK, lw=1.5, ls="--", alpha=0.4)
        ax.text((ffn_idx - 1) / 2, -0.8, sty.L["self_attention"], ha="center",
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style="italic")
        ax.text((ffn_idx + len(comp_names) - 1) / 2, -0.8, sty.L["feed_forward"], ha="center",
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, style="italic")

        cbar = ax.figure.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label("$k_{95}$", fontsize=sty.LABEL_SIZE)

    sty.generate_both(plot_anatomy, "anatomy_heatmap", figsize=sty.FIG_FULL)

---
## Fig. 2 — Decaimiento espectral por componente

**CSV necesario**: `singular_values_by_component.csv`  
Formato wide: columnas `rank_index, Query_mean, Query_std, Key_mean, Key_std, ...`  
O formato long: `rank_index, component, layer, singular_value`

In [ ]:
sv_df = load_csv(CSV_SINGULAR_VALUES, name="singular values")

if sv_df is not None:
    def plot_spectral(ax, df=sv_df):
        components = ["Query", "Key", "Value", "Attn_Output", "FFN_Inter", "FFN_Output"]
        for comp in components:
            col_mean = f"{comp}_mean"
            col_std = f"{comp}_std"
            if col_mean in df.columns:
                x = df["rank_index"] if "rank_index" in df.columns else df.index
                y = df[col_mean]
                color = sty.component_color(comp)
                lw = 2.0 if "FFN" in comp else 1.5
                ls = "-" if comp in ("Query", "Key", "FFN_Inter", "FFN_Output") else "--"
                ax.plot(x, y, color=color, lw=lw, ls=ls,
                        label=comp.replace("_", " "), alpha=0.85)
                if col_std in df.columns:
                    ax.fill_between(x, y - df[col_std], y + df[col_std],
                                    color=color, alpha=0.08)

        ax.set_xlabel(sty.L["singular_index"])
        ax.set_ylabel(sty.L["singular_magnitude"])
        ax.set_title(sty.L["t_spectral"])
        ax.legend(loc="upper right", ncol=2)
        ax.set_xlim(0, None)
        ax.set_ylim(0, None)

    sty.generate_both(plot_spectral, "spectral_fingerprint", figsize=sty.FIG_FULL)

---
## Fig. 3 — Compresión uniforme: transición de fase

In [ ]:
uni_fallback = pd.DataFrame({
    "rank":      [768,   512,   384,   256,   128,   64],
    "macro_f1":  [0.577, 0.464, 0.251, 0.025, 0.000, 0.000],
    "retention": [1.000, 0.805, 0.435, 0.043, 0.000, 0.000],
})
uni_df = load_csv(CSV_UNIFORM_RESULTS, fallback=uni_fallback, name="uniform results")

In [ ]:
def plot_transition(ax, df=uni_df):
    ranks = df["rank"].values
    retention = df["retention"].values

    ax.plot(ranks, retention, color=sty.BLUE, marker="s", markersize=8, lw=2.2,
            markeredgecolor="white", markeredgewidth=1.2, zorder=5)

    # Zones
    ax.axvspan(0, 256, alpha=0.04, color=sty.TERRA, zorder=0)
    ax.text(160, 0.88, sty.L["collapse_zone"], ha="center",
            fontsize=sty.ANNOTATION_SIZE, color=sty.TERRA_L, style="italic")
    ax.axvspan(256, 384, alpha=0.04, color=sty.SAND, zorder=0)
    ax.annotate(sty.L["phase_transition"], xy=(320, 0.24),
                fontsize=sty.ANNOTATION_SIZE, color=sty.INK_3, ha="center", style="italic")

    for r, ret, label in [(512, 0.805, "80.5%"), (384, 0.435, "43.5%"), (256, 0.043, "4.3%")]:
        offset = (10, 10) if ret > 0.1 else (10, -15)
        sty.annotate_point(ax, r, ret, label, offset=offset)

    ax.set_xlabel(sty.L["rank"])
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_uniform"])
    ax.set_xlim(0, 800)
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)
    ax.invert_xaxis()

sty.generate_both(plot_transition, "uniform_transition", figsize=sty.FIG_WIDE)

---
## Fig. 4 — Vulnerabilidad emocional bajo compresión uniforme

**CSV necesario**: `uniform_emotion_f1.csv` (filas=emociones, cols=baseline, r512, r384, r256, r128, r64)

In [ ]:
emo_df = load_csv(CSV_UNIFORM_EMOTIONS, name="uniform emotion F1")

if emo_df is not None:
    def plot_emotion_vuln(ax, df=emo_df):
        if df.columns[0].lower() in ('emotion', 'emocion'):
            df = df.set_index(df.columns[0])
        df = df.sort_values(df.columns[0], ascending=True)  # sort by baseline

        im = ax.imshow(df.values, cmap=sty.CMAP_SEQUENTIAL, aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(df.columns)))
        ax.set_xticklabels(df.columns, fontsize=sty.TICK_SIZE, rotation=45, ha="right")
        ax.set_yticks(range(len(df.index)))
        ax.set_yticklabels(df.index, fontsize=sty.TICK_SIZE)
        ax.set_title(sty.L["t_emotion_vuln"])

        for i in range(len(df.index)):
            for j in range(len(df.columns)):
                val = df.iloc[i, j]
                if not np.isnan(val):
                    color = "white" if val > 0.5 else sty.INK
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            fontsize=6, color=color)

        cbar = ax.figure.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
        cbar.set_label("F1", fontsize=sty.LABEL_SIZE)

    sty.generate_both(plot_emotion_vuln, "emotion_vulnerability", figsize=sty.FIG_TALL)

---
## Fig. 5 — Sensibilidad por componente a $r=128$

In [ ]:
comp_fallback = pd.DataFrame({
    "component": ["Query", "Key", "Value", "Attn Output", "FFN Output", "FFN Inter"],
    "f1_r128":   [0.574,  0.568,  0.395,  0.325,         0.128,        0.040],
    "retention": [0.994,  0.984,  0.685,  0.563,         0.222,        0.069],
})
comp_df = load_csv(CSV_COMP_SENSITIVITY, fallback=comp_fallback, name="component sensitivity")

In [ ]:
def plot_comp_sens(ax, df=comp_df):
    comps = df["component"].values
    retentions = df["retention"].values
    colors = [sty.component_color(c) for c in comps]

    bars = ax.barh(range(len(comps)), retentions, color=colors, height=0.55,
                   edgecolor=sty.BG, linewidth=0.8)

    for i, (comp, ret) in enumerate(zip(comps, retentions)):
        x_pos = ret + 0.015 if ret < 0.85 else ret - 0.07
        color = sty.INK_2 if ret < 0.85 else "white"
        ax.text(x_pos, i, f"{ret:.1%}", va="center", fontsize=sty.ANNOTATION_SIZE,
                color=color, weight="medium")

    ax.set_yticks(range(len(comps)))
    ax.set_yticklabels(comps)
    ax.set_xlabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_comp_sens"])
    ax.set_xlim(0, 1.12)
    sty.format_pct(ax, axis="x")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3)
    ax.grid(axis="y", visible=False)
    ax.axvline(x=0.80, color=sty.INK_3, ls=":", lw=0.7, alpha=0.5)

sty.generate_both(plot_comp_sens, "component_sensitivity", figsize=sty.FIG_WIDE)

---
## Fig. 6 — Sensibilidad por grupo de profundidad

In [ ]:
depth_fallback = pd.DataFrame({
    "group": ["Early\n(0–3)", "Middle\n(4–7)", "Late\n(8–11)"],
    "r256":  [0.854, 0.825, 0.333],
    "r128":  [0.625, 0.465, 0.000],
    "r64":   [0.370, 0.253, 0.000],
})
depth_df = load_csv(CSV_DEPTH_SENSITIVITY, fallback=depth_fallback, name="depth sensitivity")

In [ ]:
def plot_depth_sens(ax, df=depth_df):
    groups = df["group"].values
    x = np.arange(len(groups))
    width = 0.22
    rank_cols = [c for c in df.columns if c != "group"]
    rank_colors = [sty.SAGE, sty.TERRA_L, sty.TERRA]

    for i, (col, rc) in enumerate(zip(rank_cols, rank_colors)):
        vals = df[col].values
        label = f"$r{{=}}{col.replace('r', '')}$"
        ax.bar(x + i * width, vals, width, label=label, color=rc,
               edgecolor=sty.BG, linewidth=0.8)
        for j, v in enumerate(vals):
            text = f"{v:.0%}" if v > 0.01 else "0%"
            color = sty.INK_3 if v > 0.01 else sty.TERRA
            y_pos = v + 0.02 if v > 0.01 else 0.02
            ax.text(x[j] + i * width, y_pos, text, ha="center",
                    fontsize=sty.SMALL_SIZE, color=color)

    ax.set_xticks(x + width)
    ax.set_xticklabels(groups)
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_depth_sens"])
    ax.set_ylim(0, 1.05)
    sty.format_pct(ax)
    ax.legend(fontsize=sty.LEGEND_SIZE)

sty.generate_both(plot_depth_sens, "depth_sensitivity", figsize=sty.FIG_WIDE)

---
## Fig. 7 — Frontera de Pareto

In [ ]:
pareto_fallback = pd.DataFrame({
    "strategy": ["uniform_r512", "uniform_r384", "uniform_r256", "uniform_r128",
                 "adaptive_e99", "adaptive_e95", "adaptive_e90", "adaptive_e80"],
    "type":     ["uniform"]*4 + ["adaptive"]*4,
    "compression_ratio": [1.0, 0.806, 0.612, 0.418, 1.202, 1.022, 0.892, 0.721],
    "f1_retention":      [0.805, 0.435, 0.043, 0.0, 0.967, 0.850, 0.619, 0.248],
})
pareto_df = load_csv(CSV_COMPRESSION_COMPARISON, fallback=pareto_fallback, name="Pareto")

In [ ]:
def plot_pareto(ax, df=pareto_df):
    for stype in df["type"].unique():
        subset = df[df["type"] == stype].sort_values("compression_ratio")
        color = sty.strategy_color(stype)
        marker = sty.strategy_marker(stype)
        label = sty.L.get(stype, stype.capitalize())

        ax.plot(subset["compression_ratio"], subset["f1_retention"],
                color=color, lw=1.0, alpha=0.4, zorder=3)
        ax.scatter(subset["compression_ratio"], subset["f1_retention"],
                  color=color, marker=marker, s=70, edgecolor="white",
                  linewidth=0.8, zorder=5, label=label)

    ax.axhline(y=1.0, color=sty.INK_3, ls=":", lw=0.6, alpha=0.4)
    ax.axvline(x=1.0, color=sty.INK_3, ls=":", lw=0.6, alpha=0.4)
    ax.text(1.02, 0.02, sty.L["baseline_params"], fontsize=sty.SMALL_SIZE,
            color=sty.INK_3, style="italic")

    xmax = df["compression_ratio"].max() + 0.05
    if xmax > 1.0:
        ax.axvspan(1.0, xmax, alpha=0.03, color=sty.TERRA)
        ax.text((1.0 + xmax) / 2, 0.92, sty.L["expansion_zone"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, ha="center", style="italic")

    ax.legend(loc="lower right")
    ax.set_xlabel(sty.L["param_ratio"])
    ax.set_ylabel(sty.L["f1_retention"])
    ax.set_title(sty.L["t_pareto"])
    ax.set_ylim(-0.05, 1.1)
    sty.format_pct(ax)

sty.generate_both(plot_pareto, "pareto_frontier", figsize=sty.FIG_FULL)

---
## Fig. 8 — Distribución de rangos adaptativos

**CSV necesario**: `adaptive_rank_distribution.csv` (cols: threshold, component, mean_rank)

In [ ]:
arank_df = load_csv(CSV_ADAPTIVE_RANKS, name="adaptive ranks")

if arank_df is not None:
    def plot_adaptive_ranks(ax, df=arank_df):
        thresholds = sorted(df["threshold"].unique())
        components = [c for c in ["Query", "Key", "Value", "Attn_Output", "FFN_Inter", "FFN_Output"]
                      if c in df["component"].values]

        x = np.arange(len(thresholds))
        n = len(components)
        w = 0.8 / n

        for i, comp in enumerate(components):
            sub = df[df["component"] == comp].sort_values("threshold")
            color = sty.component_color(comp)
            ax.bar(x + i*w - (n-1)*w/2, sub["mean_rank"].values, w,
                   label=comp.replace("_", " "), color=color, edgecolor=sty.BG, linewidth=0.5)

        ax.axhline(y=384, color=sty.INK_3, ls="--", lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 390, sty.L["break_even_attn"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va="bottom")
        ax.axhline(y=614, color=sty.INK_3, ls="--", lw=0.7, alpha=0.5)
        ax.text(len(thresholds)-0.3, 620, sty.L["break_even_ffn"],
                fontsize=sty.SMALL_SIZE, color=sty.INK_3, va="bottom")

        ax.set_xticks(x)
        ax.set_xticklabels([f"$\\tau$={t}%" for t in thresholds])
        ax.set_ylabel(sty.L["rank_assigned"])
        ax.set_title(sty.L["t_adaptive_ranks"])
        ax.set_ylim(0, 780)
        ax.legend(fontsize=sty.SMALL_SIZE, ncol=3, loc="upper left")
        ax.grid(axis="x", visible=False)

    sty.generate_both(plot_adaptive_ranks, "adaptive_rank_distribution", figsize=sty.FIG_WIDE)

---
## Resumen

| # | Archivo | §     | Datos necesarios |
|---|---------|-------|------------------|
| 1 | `cap4_anatomy_heatmap_{es,en}.png` | §4.1 | `rank_matrix_k95.csv` |
| 2 | `cap4_spectral_fingerprint_{es,en}.png` | §4.1 | `singular_values_by_component.csv` |
| 3 | `cap4_uniform_transition_{es,en}.png` | §4.2 | hardcoded / `uniform_compression_results.csv` |
| 4 | `cap4_emotion_vulnerability_{es,en}.png` | §4.2 | `uniform_emotion_f1.csv` |
| 5 | `cap4_component_sensitivity_{es,en}.png` | §4.3 | hardcoded / `component_sensitivity.csv` |
| 6 | `cap4_depth_sensitivity_{es,en}.png` | §4.3 | hardcoded / `depth_sensitivity.csv` |
| 7 | `cap4_pareto_frontier_{es,en}.png` | §4.4 | hardcoded / `compression_comparison.csv` |
| 8 | `cap4_adaptive_rank_distribution_{es,en}.png` | §4.4 | `adaptive_rank_distribution.csv` |